In [7]:
from urllib.request import Request, urlopen
import json
import csv

url = "https://donnees.montreal.ca/api/3/action/datastore_search?resource_id=dbfc05f8-b939-4639-ae52-2e77f738e43f&limit=5&q=title:jones"

request = Request(url, headers={'User-Agent': 'MyApp/1.0'})
response = urlopen(request).read().decode('utf-8')
response = json.loads(response)

data = response["result"]["records"]



KeyboardInterrupt: 

In [104]:
"""
Find borough from location.
"""

import json
import shapely
from pathlib import Path
from pyproj import Transformer


class BoroughIdentifier():
    def __init__(self):
        self.boroughs = [
            "Ahuntsic-Cartierville", 
            "Anjou", 
            "Côte-des-Neiges-Notre-Dame-de-Grâce", 
            "Lachine",
            "LaSalle",
            "Le Plateau-Mont-Royal",
            "Le Sud-Ouest",
            "L'Île-Bizard-Sainte-Geneviève",
            "Mercier-Hochelaga-Maisonneuve",
            "Montréal-Nord",
            "Outremont",
            "Pierrefonds-Roxboro",
            "Rivière-des-Prairies-Pointe-aux-Trembles",
            "Rosemont-La Petite-Patrie",
            "Saint-Laurent",
            "Saint-Léonard",
            "Verdun",
            "Ville-Marie",
            "Villeray-Saint-Michel-Parc-Extension"
        ]

        self.borough_polygons = self._initialize_limits()


    def _initialize_limits(self):
        "Creates a dictionary of the borough shapes."
        path = Path.cwd().resolve().parent / "data" / "borough_limits.geojson"
        with open(path, "r", encoding = "utf-8") as file:
            data = json.load(file)

        polygons = {}
        for borough in data["features"]:
            borough_name = borough["properties"]["NOM"]
            if borough_name in self.boroughs:
                borough_string = json.dumps(borough["geometry"]) # Need json dumps because from_geojson expects a string, not the object
                polygons[borough_name] = shapely.from_geojson(borough_string) 

        if len(list(polygons.keys())) != len(self.boroughs):
            print("ERROR IN INITIALIZATION OF BOROUGH LIMITS.")

        return polygons

    def convert_WSG84_to_NAD83(self, longitude_x, latitude_y):
        "Converts WSG84 coordinates to the NAD83 system of the boroughs."
        transformer = Transformer.from_crs("EPSG:4326", "EPSG:32188", always_xy = True)
        (new_x, new_y) = transformer.transform(longitude_x, latitude_y)
        return new_x, new_y
        
    def match_WSG84_to_borough(self, longitude_x, latitude_y):
        "Finds the borough the WSG84 location belongs to."
        NAD83_x, NAD83_y  = self.convert_WSG84_to_NAD83(longitude_x, latitude_y)
        for borough in self.boroughs:
            if shapely.contains_xy(self.borough_polygons[borough], x = NAD83_x, y = NAD83_y):
                return borough
        return None

    def match_NAD83_to_borough(self, longitude_x, latitude_y):
        "Finds the borough the NAD83 location belongs to."
        for borough in self.boroughs:
            if shapely.contains_xy(self.borough_polygons[borough],  x = longitude_x, y = latitude_y):
                return borough
        return None

In [105]:
identifer = BoroughIdentifier()
borough = identifer.match_WSG84_to_borough(-73.626778, 45.56778)
print(borough)
borough = identifer.match_NAD83_to_borough(294904.159001211, 5047548.92099452)
print(borough)

Villeray-Saint-Michel-Parc-Extension
Villeray-Saint-Michel-Parc-Extension
